### DistilBERT and CrisisMMD

This set of experiments evaluates DistilBERT's training configuration and baseline performance ahead of introducing DP-SGD, repeating the calibration work already done for MobileNetV3.

**Learning rate sweep.** With privacy disabled, DistilBERT was trained on the Emotion dataset across learning rates from 1e-7 to 1. Accuracy and loss both improved steadily up to 0.001, but at 0.01 and above training collapsed, the loss curves flatten immediately and accuracy sticks near the random-guess baseline, probably indicating that the optimizer is overshooting. Learning rate 0.0001 was selected going forward, giving strong final accuracy 87% with a safe margin from the collapse point.

**Layer-freezing sweep.** Holding learning rate fixed at 0.0001, unfreezing layer 1 and above converges to 88-90% accuracy on Emotion. A fully frozen backbone (only the classifier head trainable, configuration -1) gives around 38%, and configuration 0 (classifier head fully trainable, backbone frozen) results at near 52%. This shows the pretrained DistilBERT backbone needs at least one unfrozen transformer layer.

**CrisisMMD 2.0** - a multimodal Twitter dataset collected during major natural disasters (only the text modality is used in this work). It is pre-split by its authors into 13,608 training messages and 2,237 test messages, and defines eight informativeness labels: affected_individuals, infrastructure_and_utility_damage, injured_or_dead_people, missing_or_found_people, not_humanitarian, other_relevant_information, rescue_volunteering_or_donation_effort, vehicle_damage.

**CrisisMMD text modality.** The same sweep was repeated on the text component of CrisisMMD 2.0, repllacing Emotion with the target disaster-relevant dataset. Configurations with layer 1 and above unfrozen converge to a lower ceiling of 67-69% accuracy - noticeably below the 88-90% reached on Emotion. This is expected given the nature of the data: unlike Emotion's clean, single-topic sentiment labels, CrisisMMD is real Twitter traffic collected during live disasters, and a part of it is noisy, off-topic, or ambiguous. The label set itself also captures that ambiguity (e.g. distinguishing "other_relevant_information" from "not_humanitarian" is an uncertain boundary even for a human), so a lower ceiling here reflects the dataset structure rather than a model issue.

Separately, the loss curves for the configurations 2 and above start climbing again after round 8 even as accuracy stays flat, which is a mild overfitting signature.

##### page break

In [ ]:
from matplotlib import pyplot
import pandas
def flwr_accuracy(file, start=0, key="accuracy"):
       df = pandas.read_csv(file + ".csv")
       pyplot.figure(figsize=(8,4))
       for id_val, g in df.groupby("id", sort=False):
              g = g.sort_values("round").tail(len(g) - start)
              pyplot.plot(g["round"], g[key], label=str(id_val))

       pyplot.xlabel("round")
       pyplot.ylabel(key)
       pyplot.grid(True, alpha=0.3)
       pyplot.legend()
       pyplot.tight_layout()
       pyplot.show()

##### page break

### DistilBERT performance by learning rate
noise_multiplier: 0, tune_layers: 1, batch_size: 32

DistilBERT model on the Emotion dataset (16,000 training / 2,000 test texts) with privacy noise disabled. Accuracy and loss both improved steadily as the learning rate increased from 1e-7 to 0.001, but at 0.01 and above the model stopped learning, producing effectively random output. Learning rate 0.0001 was selected for the following experiments as it gives good results and far enough from the collapse value.

In [ ]:
flwr_accuracy("bert_clear_learning")

In [ ]:
flwr_accuracy("bert_clear_learning", key="loss")

##### page break

### DistilBERT performance by frozen layers
noise_multiplier: 0, learning_rate: 0.0001, batch_size: 32

With layers 2, 3 and above unfrozen, the model converges to 88-90% accuracy on the Emotion dataset. However, a fully frozen backbone with only the classifier head trainable, was unable to provide result better than 60%, indicating that at least one unfrozen backbone layer is necessary for DistilBERT performance.

In [ ]:
flwr_accuracy("bert_clear_layers")

In [ ]:
flwr_accuracy("bert_clear_layers", key="loss")

##### page break

### DistilBERT performance on CrisisMMD
noise_multiplier: 0, learning_rate: 0.0001, batch_size: 32

In this test the Emotion dataset was replaced with the text modality of CrisisMMD 2.0, a state of the art dataset of social media messages and images from various disaster types. It defines 8 labels and is pre-split by its authors into 13,608 training and 2,237 test texts.

Configurations with layer 1 and above unfrozen converge to 67-69% accuracy. Models with more unfrozen layers show a higher rise in loss value during later training rounds.

In [ ]:
flwr_accuracy("crisismmd_layers")

In [ ]:
flwr_accuracy("crisismmd_layers", key="loss")